download data - brazilian e-commerce

In [ ]:
# kaggle installation

!pip install kaggle

   ---------------------------------------- 0.0/89.7 kB ? eta -:--:--
   ------------- -------------------------- 30.7/89.7 kB 1.4 MB/s eta 0:00:01
   ------------------ --------------------- 41.0/89.7 kB 667.8 kB/s eta 0:00:01
   ------------------------------------ --- 81.9/89.7 kB 657.6 kB/s eta 0:00:01
   ---------------------------------------- 89.7/89.7 kB 635.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/170.5 kB ? eta -:--:--
   --------- ------------------------------ 41.0/170.5 kB ? eta -:--:--
   ---------------------------- ----------- 122.9/170.5 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 170.5/170.5 kB 1.7 MB/s eta 0:00:00


In [ ]:
import os # to handle environment variables and file manipulation
from kaggle.api.kaggle_api_extended import KaggleApi # to access the Kaggle API

os.environ['KAGGLE_USERNAME'] = 'claricematos'
os.environ['KAGGLE_KEY'] = 'KGAT_215b3eafa49cf7096d2fa014eaa2d765'

dest = r'C:\Users\andra\OneDrive\00-data-science\portifolio\olist-churn-risk\data\raw'
os.makedirs(dest, exist_ok=True) # creates the destination directory if it doesn't exist

api = KaggleApi() # initializes the Kaggle API
api.authenticate() # authenticates the user using the provided credentials   
api.dataset_download_files('olistbr/brazilian-ecommerce', path=dest, unzip=True) # downloads and extracts the dataset from Kaggle to the specified directory

print("Download completed!")

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
✅ Download concluído!


In [ ]:
# transforming data to pandas dataframe

import pandas as pd

orders = pd.read_csv('data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('data/raw/olist_order_items_dataset.csv')
order_reviews = pd.read_csv('data/raw/olist_order_reviews_dataset.csv')
customers = pd.read_csv('data/raw/olist_customers_dataset.csv')
products = pd.read_csv('data/raw/olist_products_dataset.csv')
payments = pd.read_csv('data/raw/olist_order_payments_dataset.csv')

print("Load data!")

✅ Load data!


In [ ]:
# general overview of the datasets

dfs = {                 # dictionary to hold the dataframes
    'order_items': order_items,
    'order_reviews': order_reviews,
    'customers': customers,
    'products': products,
    'payments': payments
}

for name, df in dfs.items():        # loop through each dataframe in the dictionary
    print(f'\n{"="*50}')
    print(f'📦 {name.upper()}')
    print(f'Shape: {df.shape}')
    print(f'\nColunas: {list(df.columns)}')
    print(f'\nMissing values:')
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing[missing > 0].any() else "Nenhum")


📦 ORDERS
Shape: (99441, 8)

Colunas: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Missing values:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

📦 ORDER_ITEMS
Shape: (112650, 7)

Colunas: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Missing values:
Nenhum

📦 ORDER_REVIEWS
Shape: (99224, 7)

Colunas: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Missing values:
review_comment_title      87656
review_comment_message    58247
dtype: int64

📦 CUSTOMERS
Shape: (99441, 5)

Colunas: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Missing values:
Nenhum

📦 PR

In [ ]:
# order status analysis

print(orders['order_status'].value_counts())
print(f"\nTotal de pedidos entregues: {(orders['order_status'] == 'delivered').sum()}")
print(f"Proporção entregues: {(orders['order_status'] == 'delivered').mean()*100:.1f}%")

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Total de pedidos entregues: 96478
Proporção entregues: 97.0%


In [23]:
# orders reviews analysis

print(order_reviews['review_score'].value_counts().sort_index())
print(f"\nAvaliações <= 2: {(order_reviews['review_score'] <= 2).sum()} "
      f"({(order_reviews['review_score'] <= 2).mean()*100:.1f}%)")

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Avaliações <= 2: 14575 (14.7%)


In [ ]:
# checking delay between estimated and actual delivery

# Convertendo datas
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# Filtering only delivered items
delivered = orders[orders['order_status'] == 'delivered'].copy()

# Calculating delay
delivered['delivery_delay_days'] = (
    delivered['order_delivered_customer_date'] -
    delivered['order_estimated_delivery_date']
).dt.days

print(f"Pedidos com atraso: {(delivered['delivery_delay_days'] > 0).sum()} "
      f"({(delivered['delivery_delay_days'] > 0).mean()*100:.1f}%)")

print(f"\nEstatísticas do atraso:")
print(delivered['delivery_delay_days'].describe())

Pedidos com atraso: 6534 (6.8%)

Estatísticas do atraso:
count    96470.000000
mean       -11.875889
std         10.182105
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64


In [ ]:
# bulding a simple churn definition based on reviews and delivery delay

# Merge orders + reviews
df = delivered.merge(
    order_reviews[['order_id', 'review_score']],
    on='order_id',
    how='left'
)

# Defining churn_risk
df['churn_risk'] = (
    (df['review_score'] <= 2) |
    (df['delivery_delay_days'] > 0)
).astype(int)

print("Target distribution:")
print(df['churn_risk'].value_counts())
print(f"\nProportion churn_risk = 1: {df['churn_risk'].mean()*100:.1f}%")

# Checking orders without evaluation
print(f"\nOrders without evaluation: {df['review_score'].isnull().sum()}")

Distribuição do target:
churn_risk
0    82098
1    14909
Name: count, dtype: int64

Proporção churn_risk = 1: 15.4%

Pedidos sem avaliação: 646


In [ ]:
# saving the base for future steps

os.makedirs('data/processed', exist_ok=True) 
df.to_csv('data/processed/orders_with_target.csv', index=False)
print("✅ Base salva em data/processed/orders_with_target.csv")

✅ Base salva em data/processed/orders_with_target.csv
